# 02. Predictive Modelling & Training Evaluation (Seen Data)
This notebook implements the supervised machine learning pipeline for binary classification to predict English Premier League match outcomes (`HomeWin` = 1 for a home victory, `HomeWin` = 0 for a draw or away victory).

### Objectives:
1. Load the processed Premier League cohort (2014–2024).
2. Construct the pre-match feature matrix ($X$) and ground truth target vector ($y$), computing the Elo rating differential (`EloDiff`).
3. Apply a stratified 80/20 train-test split (`train_test_split`) with a fixed seed for strict experimental reproducibility.
4. Address missing values and scale numerical features ensuring zero temporal and partition data leakage.
5. Fit a majority-class baseline and three representative supervised classifiers using default hyperparameters.
6. Evaluate model discrimination on seen training data to empirically inspect memorisation and diagnose overfitting.

In [1]:
import os
import pandas as pd
import numpy as np

# 1. Cargar el dataset procesado
data_path = os.path.join('..', 'data', 'processed', 'premier_league_2014_2024.csv')
df = pd.read_csv(data_path)

# 2. Ingeniería de características: calcular la diferencia de rating Elo
df['EloDiff'] = df['HomeElo'] - df['AwayElo']

# 3. Selección de predictores pre-partido (X) y variable objetivo (y)
feature_cols = [
    'EloDiff', 'HomeElo', 'AwayElo',
    'Form5Home', 'Form5Away',
    'OddHome', 'OddDraw', 'OddAway'
]

X = df[feature_cols].copy()
y = df['HomeWin'].copy()

# 4. Verificación de dimensiones e integridad
print(f"Total de partidos cargados: {len(df)}")
print(f"Dimensiones de X (predictores): {X.shape}")
print(f"Dimensiones de y (target): {y.shape}")
print("\nComprobación de valores nulos en X:")
print(X.isna().sum())

Total de partidos cargados: 4178
Dimensiones de X (predictores): (4178, 8)
Dimensiones de y (target): (4178,)

Comprobación de valores nulos en X:
EloDiff      2
HomeElo      1
AwayElo      1
Form5Home    0
Form5Away    0
OddHome      0
OddDraw      0
OddAway      0
dtype: int64


### 1. Stratified Partitioning and Leakage-Safe Preprocessing
We perform an 80/20 train-test split using `train_test_split`, stratified across the target variable (`stratify=y`) with `random_state=42` to guarantee identical class balance across both partitions.

To prevent data leakage:
1. Two missing values in the historical Elo columns are imputed using the training median learned exclusively from `X_train`.
2. A `StandardScaler` is fitted solely on the training partition (`X_train`) via `fit_transform` and subsequently applied to `X_test` via `transform`.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Partición estratificada 80% train (3.342 partidos), 20% test (836 partidos)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Imputación con la mediana aprendida SOLO de X_train (evita fuga)
for col in ['HomeElo', 'AwayElo', 'EloDiff']:
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)

# 3. Escalado con StandardScaler ajustado SOLO en train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Verificaciones
print(f"Partición completada con éxito:")
print(f"Partidos de entrenamiento (X_train): {X_train.shape[0]}")
print(f"Partidos de prueba ciegos (X_test):   {X_test.shape[0]}")
print(f"\nNulos restantes en X_train: {X_train.isna().sum().sum()}")
print(f"Nulos restantes en X_test:  {X_test.isna().sum().sum()}")
print(f"\nDistribución del target en X_train (%):")
print(y_train.value_counts(normalize=True).round(3) * 100)

Partición completada con éxito:
Partidos de entrenamiento (X_train): 3342
Partidos de prueba ciegos (X_test):   836

Nulos restantes en X_train: 0
Nulos restantes en X_test:  0

Distribución del target en X_train (%):
HomeWin
0    55.1
1    44.9
Name: proportion, dtype: float64


### 2. Default Model Training and Evaluation on Seen Data (Training Set)
Four distinct architectures are fitted on `X_train` using their default hyperparameters:
* **Naive Baseline (`DummyClassifier`):** Uses the `most_frequent` strategy to establish the empirical performance floor (predicting class 0 for all matches).
* **Logistic Regression (`LogisticRegression`):** A regularised linear probabilistic classifier fitted on standardised features (`X_train_scaled`).
* **Decision Tree (`DecisionTreeClassifier`):** An unpruned tree (`max_depth=None`) trained to evaluate sample memorisation and overfitting.
* **Random Forest (`RandomForestClassifier`):** An ensemble bagging model trained with 100 default estimators.

Performance metrics are evaluated on the seen training data to inspect model capacity and diagnose overfitting.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# 1. Instanciar modelos con parámetros por defecto y semilla fija 42
models = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree (Unpruned)": DecisionTreeClassifier(random_state=42),
    "Random Forest (Default)": RandomForestClassifier(random_state=42)
}

# 2. Entrenar y evaluar en datos vistos (X_train)
seen_results = []

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_train_scaled)
        y_prob = model.predict_proba(X_train_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_train)
        y_prob = model.predict_proba(X_train)[:, 1] if hasattr(model, "predict_proba") else y_pred

    seen_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_train, y_pred),
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1-Score": f1_score(y_train, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_train, y_prob) if name != "Baseline (Dummy)" else 0.5
    })

# 3. Presentar tabla comparativa en consola
df_seen = pd.DataFrame(seen_results)
print("--- EVALUATION ON SEEN DATA (TRAINING SET) ---")
print(df_seen.round(3).to_string(index=False))

--- EVALUATION ON SEEN DATA (TRAINING SET) ---
                   Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
        Baseline (Dummy)     0.551      0.000   0.000     0.000    0.500
     Logistic Regression     0.683      0.689   0.534     0.602    0.737
Decision Tree (Unpruned)     1.000      1.000   1.000     1.000    1.000
 Random Forest (Default)     1.000      1.000   1.000     1.000    1.000


### 3. Visualising Model Performance on Seen Data
To illustrate the contrast between constrained linear learning and unconstrained tree memorisation, we generate a grouped bar chart comparing Accuracy, F1-Score, and ROC-AUC across all four baseline classifiers on the seen training partition (`X_train`).